# 02 — Data Cleaning and Cycle Validation

**Goal:** Build/update the cycle manifest (one row per cycle: condition, team member, start/end
time, notes) and flag + exclude any cycle that got contaminated — rain in a "sheltered" outdoor
pot, an accidental keypad override mid-cycle, a malfunctioning sensor. Output is the clean list
of valid cycles everything downstream uses.

**This notebook is where you exercise judgment.** The exclusions cell below is a placeholder —
replace it with what actually happened during your real collection.

In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
PREPROCESSED_DIR = Path("../data/preprocessed")
PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## Load (or initialize) the cycle manifest

In [2]:
manifest_path = PREPROCESSED_DIR / "cycle_manifest_DUMMY.csv"

if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print(f"Loaded existing manifest: {len(manifest)} cycles")
else:
    rows = []
    for f in sorted(RAW_DIR.glob("*.csv")):
        df = pd.read_csv(f, parse_dates=["timestamp"])
        cycle_id = f.stem.replace("_DUMMY", "")
        rows.append({
            "cycle_id": cycle_id,
            "condition": df["condition"].iloc[0],
            "team_member": "UNKNOWN",
            "start_time": df["timestamp"].iloc[0],
            "end_time": df["timestamp"].iloc[-1],
            "duration_hours": len(df),
            "excluded": False,
            "exclusion_reason": "",
            "notes": "",
        })
    manifest = pd.DataFrame(rows)
    print(f"Built new manifest: {len(manifest)} cycles")

manifest

Loaded existing manifest: 4 cycles


,cycle_id,condition,team_member,start_time,end_time,duration_hours,excluded,exclusion_reason,notes
0,indoor_cycle01,indoor,DUMMY_GENERATOR,2026-07-01 09:00:00,2026-07-02 12:00:00,28,False,NaN,Synthetic placeholder data for pipeline testin...
1,indoor_cycle02,indoor,DUMMY_GENERATOR,2026-07-02 15:00:00,2026-07-03 18:00:00,28,False,NaN,Synthetic placeholder data for pipeline testin...
2,outdoor_cycle01,outdoor,DUMMY_GENERATOR,2026-07-03 21:00:00,2026-07-04 15:00:00,19,False,NaN,Synthetic placeholder data for pipeline testin...
3,outdoor_cycle02,outdoor,DUMMY_GENERATOR,2026-07-04 18:00:00,2026-07-05 12:00:00,19,False,NaN,Synthetic placeholder data for pipeline testin...


## Flag cycles for exclusion

Replace this cell's dict with your team's real field notes. Common real exclusion reasons for
this project: rain got under a "sheltered" outdoor cover, keypad override accidentally pressed
during real collection (not demo), sensor flat-lined or jumped impossibly, pot got bumped/moved.

In [3]:
# EXAMPLE placeholder — edit with your team's real cycle exclusions.
# Format: {cycle_id: "reason"}
exclusions = {
    # "outdoor_cycle03": "rain got under the shelter cover, moisture spiked mid-cycle",
    # "indoor_cycle05": "keypad override accidentally triggered by teammate",
}

manifest["excluded"] = manifest["cycle_id"].isin(exclusions.keys())
manifest["exclusion_reason"] = manifest["cycle_id"].map(exclusions).fillna("")

print(f"Excluded {manifest['excluded'].sum()} of {len(manifest)} cycles")
manifest[manifest["excluded"]]

Excluded 0 of 4 cycles


,cycle_id,condition,team_member,start_time,end_time,duration_hours,excluded,exclusion_reason,notes


## Save the validated manifest

In [4]:
manifest.to_csv(PREPROCESSED_DIR / "cycle_manifest_validated.csv", index=False)
print("Saved cycle_manifest_validated.csv")

valid_cycles = manifest[~manifest["excluded"]]["cycle_id"].tolist()
print(f"\n{len(valid_cycles)} valid cycles proceeding to feature engineering:")
for c in valid_cycles:
    print(" -", c)

Saved cycle_manifest_validated.csv

4 valid cycles proceeding to feature engineering:
 - indoor_cycle01
 - indoor_cycle02
 - outdoor_cycle01
 - outdoor_cycle02


**Next step:** `03_feature_engineering.ipynb` — compute moisture trend and hour-of-day for every valid cycle.